# Belief updating: reproduce, correct, transfer

**Goal.** Recompute the gratitude notebook's selected numerical cells, inspect
implementation discrepancies, and follow the corrected learning parameters
into the scene model. This companion runs the canonical GNN study, not a
screenshot or a cached notebook result. All agents are synthetic.

Run from this repository with the project environment (`uv sync --frozen --extra dev`).
`uv run python scripts/run_pipeline.py` executes and exports all companions.
Source notebooks are preserved unchanged in `data/sources`; their original
Colab installation and interactive interfaces are outside this reconstruction.
The [notebook audit](../docs/notebook-audit.md) records cell-level scope.

In [ ]:
import sys
from thanks_for_nothing.notebooks import notebook_context
context = notebook_context()
from dataclasses import asdict
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
from pprint import pprint
import numpy as np
from IPython.display import Image, Markdown, display
from thanks_for_nothing.artifacts import write_record
from thanks_for_nothing.experiments import run_learning_experiment
print({name: version(name) for name in (
    'thanks-for-nothing', 'generalized-notation-notation', 'realizing-emptiness')})
print('Interpreter:', sys.executable)
root = context.root
if root is None:
    raise RuntimeError('Run this companion from the repository root or notebooks directory')

## 1. Execute the original 1,000-agent numerical experiment

The adapter retains the original equations, action conventions and aliasing.
It binds both RNGs and runs numerical cells in source order. Reproducible
reconstruction does not turn an unnormalized weight matrix into a probability
channel or repair a baseline that is later mutated through a view.

In [ ]:
from thanks_for_nothing.source_reconstruction import (
    build_source_audit, run_source_reconstruction)
from thanks_for_nothing.analysis_design import load_analysis
analysis = load_analysis(context.specifications)
original = run_source_reconstruction(root=root, analysis=analysis)
source_audit = build_source_audit(root, analysis=analysis)
print('Luckiness grid:', len(original.luckiness),
      float(original.luckiness[0]), float(original.luckiness[-1]))
print('Aliased baseline:', original.shares_baseline_memory)
print('Before/after assessment priors equal:',
      np.array_equal(original.task_priors_before, original.task_priors_after))
pprint(source_audit['controls'])
assert len(original.luckiness) == analysis.reconstruction_agents
assert original.shares_baseline_memory

## 2. Train genuinely paired conditions and freeze their channels

Every agent shares an exposure stream across no recall, positive recall,
matched random recall and valuation-only absence appreciation. Assessment is
generated independently and never updates a transferred matrix. The `pre` and
`post` channels below are frozen before either assessment. Only GNN resources
specify the canonical model and experimental design.

In [ ]:
learning = run_learning_experiment(specification_dir=context.specifications)
study = learning.study
print('Independent paired agents:', len({row.paired_id for row in study.rows}))
print('Agent-condition records:', len(study.rows))
pprint(study.controls)
assert all(study.controls.values())
matrices = np.asarray([(row.pre, row.post) for row in learning.parameters])
from thanks_for_nothing.specifications import load_model
expected_shape = np.asarray(load_model('gratitude', context.specifications).tensors.A).shape
assert matrices.shape == (len(study.rows), 2, *expected_shape)
assert np.allclose(matrices.sum(axis=2), 1)
assert (matrices > 0).all()
print('Frozen matrices:', matrices.shape)
pprint(asdict(learning.parameters[0]))

## 3. Separate optimism-task changes from predictive consequences

Lower Brier and log loss is better prediction. Brier combines calibration
and resolution; it does not isolate calibration. The primary tests
retain their existing Holm family. Paired bootstrap units are complete
agents within fixed worlds, not trials or recall events.

In [ ]:
from thanks_for_nothing.statistics import build_statistical_report
from thanks_for_nothing.analysis_design import load_analysis
analysis = load_analysis(context.specifications)
report = build_statistical_report(
    study.rows, seed=study.design.seed + analysis.statistics_seed_offset,
    bootstrap_draws=study.design.bootstrap_draws,
    permutation_draws=study.design.permutation_draws, analysis=analysis)
for row in report.contrasts:
    print(row.endpoint, row.first, 'minus', row.second,
          'mean=', row.estimate.mean_difference,
          'bootstrap t=', asdict(row.estimate.studentized),
          'Holm p=', row.holm_p_value, 'MC decision=', row.decision)

## 4. Inspect world dependence instead of choosing a favorable stratum

The grid is a fixed estimand, not an estimated population distribution.
The convex envelope holds stratum means fixed and varies their weights.
It is neither an uncertainty interval nor additional simulation evidence.

In [ ]:
for row in report.heterogeneity.weighting:
    print(row.endpoint, row.first, 'minus', row.second,
          'equal-grid=', row.equal_grid.mean,
          'weight envelope=', (row.convex_min, row.convex_max),
          'leave-one-world-out=', (row.leave_one_out_min, row.leave_one_out_max))

## 5. Compare two explicit learning-to-gaze bridges

The source notebooks contain no executable connection. The primary new bridge
embeds the actual learned likelihood into the declared scene. The readout
comparator mixes fixed gaze templates using the assessed world-well readout.
These are different mechanisms and can reverse the direction of the effect.
Their transfer contrasts are exploratory pointwise intervals, outside the
primary family. The gaze companion derives the policy and scene-risk score.
The gaze companion also evaluates one exploratory joint region across all
declared gaze contrasts, keeping these original pointwise intervals and the
primary family unchanged. Joint adjustment does not remove
distributional undercoverage or establish empirical efficacy.


In [ ]:
from thanks_for_nothing.gaze import build_gaze_audit
from thanks_for_nothing.gaze_findings import gaze_main_findings
gaze = build_gaze_audit(learning, root=root, specification_dir=context.specifications)
assert gaze.all_controls_pass
display(Markdown(gaze_main_findings(gaze)))
for row in gaze.contrasts:
    if row.experiment == 'transfer':
        print(row.context, row.condition, 'minus', row.reference, row.endpoint,
              row.intervals.mean, asdict(row.intervals.studentized))

In [ ]:
from thanks_for_nothing.gaze_summary_figures import gaze_transfer_summary
with TemporaryDirectory(prefix='tfn-learning-companion-') as directory:
    destination = Path(directory)
    write_record(destination / 'data/study.json', study)
    write_record(destination / 'data/gaze.json', gaze)
    figure = gaze_transfer_summary(gaze, destination / 'figures')
    display(Image(filename=str(destination / 'figures/gaze_transfer_summary.png'),
                  alt=figure.alt))
    print(figure.alt)

## Takeaways and next experiments

Preserving the original behavior, correcting the statistical design and
declaring a transfer map answer different questions. Positive-looking
probability is not predictive improvement. The absence condition is an
exact mechanism-imposed learning null. No result establishes human gratitude,
therapeutic benefit or contemplative realization.

To explore alternatives, copy the packaged GNN declarations, edit the
learning or gaze parameters and pass `specification_dir` to the producers.
Keep the original declarations and report all affected endpoints. The
[gaze companion](gaze_emptiness.ipynb) and [model contract](../docs/model-contract.md)
explain structural access, scene changes and semantic controls.